### **get into the right path**

In [ ]:
%cd /home/ahmed348/TCNN_repo

### **imports**

In [ ]:
import os, torch
import numpy as np
from torch import nn, optim
from tcnn_utils import get_fold_i, sEEG_EvalDataset
from machine_learning import sEEG_Dataset, train_model, evaluate_model, save_loss_plot
from machine_learning import TemporalCNN_deep
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
feat_path = f'/scratch/gilbreth/ahmed348/Dutch_dataset_features/TCNN_folder/FEATURES/features_HGA_ONLY'
pts = ['sub-%02d'%i for i in range(1,11)]
print(DEVICE)

In [ ]:
!nvidia-smi

### **generate the line-plots**

In [ ]:
k = 1  # arbitrary fold number
sub_list = list(range(10))

def load_channel_labels(txt_path):
    df = pd.read_csv(txt_path, sep="\t")
    labels = df["description"].tolist()
    return labels

# -------------------------------------------------
# Create a single figure with ten subplots
# -------------------------------------------------
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# subplot_idx = 0  # keeps track of (a), (b)
fig, axes = plt.subplots(5, 2, figsize=(14, 18))
axes = axes.flatten()  # easier indexing
subplot_idx = 0

for pNr, pt in enumerate(pts):
    if pNr not in sub_list:
        continue

    print(f'pnr : {pNr}')
    spectrogram = np.load(os.path.join(feat_path, f'{pt}_spec.npy'))
    eeg = np.load(os.path.join(feat_path, f'{pt}_feat.npy'))
    va_labels = np.load(os.path.join(feat_path, f'{pt}_va.npy'))

    print(f'patient number : {pNr+1}, fold number : {k}') 
    X_train, y_train, va_labels_train, X_test, y_test, \
        va_labels_test = get_fold_i(eeg, spectrogram, va_labels, 10, k)

    channel_means_train = np.mean(X_train, axis=0)
    channel_stds_train = np.std(X_train, axis=0)
    channel_means_test = np.mean(X_test, axis=0)
    channel_stds_test = np.std(X_test, axis=0)

    X_train = (X_train - channel_means_train) / channel_stds_train
    X_test = (X_test - channel_means_test) / channel_stds_test

    X_train = torch.from_numpy(X_train)
    y_train = torch.from_numpy(y_train)
    X_test = torch.from_numpy(X_test)
    y_test = torch.from_numpy(y_test)

    test_dataset = sEEG_EvalDataset(X_test, y_test, va_labels_test, 6)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    input_channels = X_train.shape[-1]
    output_dim = 80

    model = TemporalCNN_deep(
        input_channels, output_dim, [15, 15, 15], [1, 3, 5], 128, 0
    ).to(DEVICE)

    criterion = nn.MSELoss()
    saved_weights_path = f'/scratch/gilbreth/ahmed348/Dutch_dataset_features/TCNN_folder/SAVED_WEIGHTS/weights_HGA_ONLY_135_dil_151515_ker_7500_T_6/weights_{pt}_fold_{k}.pth'
    model.load_state_dict(torch.load(saved_weights_path))
    model.eval()

    all_channel_scores = []

    for x, y, _ in test_loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        x.requires_grad_(True)

        y_hat = model(x)
        loss = criterion(y_hat, y)

        model.zero_grad()
        loss.backward()

        grads = x.grad.abs()
        channel_scores = grads.mean(dim=1) ## aggregate the time dimension
        all_channel_scores.append(channel_scores.detach())

    all_channel_scores = torch.cat(all_channel_scores, dim=0)
    final_channel_importance = all_channel_scores.mean(dim=0) ## aggregate across batches

    txt_path = f"/scratch/gilbreth/ahmed348/public_datasets/Dutch_Dataset/sub-{pNr+1:02}/ieeg/sub-{pNr+1:02}_task-wordProduction_channels.tsv"
    channel_labels = load_channel_labels(txt_path)

    channel_importance = final_channel_importance.cpu().numpy()
    sorted_indices = np.argsort(-channel_importance)
    sorted_labels = [channel_labels[i] for i in sorted_indices]
    sorted_importance = channel_importance[sorted_indices]

    K = 15
    sorted_importance_norm = sorted_importance / sorted_importance.sum()
    topk_importance = sorted_importance_norm[:K]
    topk_labels = sorted_labels[:K]

    # -------------------------------------------------
    # Plot into subplot
    # -------------------------------------------------
    ax = axes[subplot_idx]
    ax.bar(np.arange(K), topk_importance)
    ax.set_xticks(np.arange(K))
    ax.set_xticklabels(topk_labels, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("$I_c$ (normalized)")
    ax.set_title(f"Sub-{pNr+1:02}")

    ax.text(
    -0.12, 1.08,
    f"({chr(97 + subplot_idx)})",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top"
    )
    subplot_idx += 1

# -------------------------------------------------
# Final layout & save
# -------------------------------------------------
plt.tight_layout()
# plt.savefig(
#     "/home/ahmed348/TCNN_repo/figures/channel_importance_combined_all_ten.pdf",
#     bbox_inches="tight"
# )
plt.show()

### **Collect the sensory-motor scores for top k=15 channels**

### **Collect the causal-anti_causal scores from .pkl files**

### **Generate the scatter plot**